# 03 Benders —— 子问题对偶与最优性割的完整推导

## 分解结构

主问题选列 $y_p\in\{0,1\}$（是否允许使用列 $p$），子问题在 $y$ 允许的列上解连续覆盖 LP。
列成本含在子问题中，故主问题目标只取 $\min\theta$（割把成本信息带回主问题）。

## 子问题 SP(y) 与它的对偶

$$\min_{x,s}\ \sum_p c_px_p+M\sum_i s_i \quad\text{s.t.}\quad \sum_p a_{ip}x_p+s_i\ge 1\ (\pi_i\ge 0),\ \sum_p x_p\le K\ (\mu\le 0),\ 0\le x_p\le y_p\ (\sigma_p\le 0),\ s_i\ge 0$$

按 02 的对偶规则逐条推导：

- 覆盖约束（≥）→ $\pi_i\ge 0$；车辆数（≤）→ $\mu\le 0$；上界 $x_p\le y_p$（≤）→ $\sigma_p\le 0$（目标系数 $y_p$）
- 变量 $x_p\ge 0$ → 对偶约束 $\sum_i a_{ip}\pi_i+\mu+\sigma_p\le c_p$
- 变量 $s_i\ge 0$（目标系数 $M$）→ 对偶约束 $\pi_i\le M$

$$\max_{\pi,\mu,\sigma}\ \sum_i\pi_i+K\mu+\sum_p\sigma_p y_p \quad\text{s.t.}\quad \sum_i a_{ip}\pi_i+\mu+\sigma_p\le c_p,\ \pi_i\le M,\ \pi_i\ge 0,\ \mu\le 0,\ \sigma_p\le 0$$

## Benders 最优性割的推导与有效性

设 $V(y)=SP(y)$ 的最优值，对偶可行点 $(\pi^k,\mu^k,\sigma^k)$ 给出对**任意** $y$ 都成立的下界：

$$V(y)\ \ge\ \sum_i\pi_i^k+K\mu^k+\sum_p\sigma_p^k y_p \qquad(\text{弱对偶：对偶可行值 ≤ 原最优值})$$

取 $\lambda_p=-\sigma_p\ge 0$，主问题引入 $\theta$（$V(y)$ 的逐次下界近似），割：

$$\theta+\sum_p\lambda_p^k y_p\ \ge\ \sum_i\pi_i^k+K\mu^k$$

- **有效性**：对偶可行点对任意 $y$ 都给出 $V(y)$ 的下界，割不过切。
- **收敛**：主问题 $\min\theta$ 被割逐次抬高，收敛到 $\min_y V(y)$ = 候选池的 LP 松弛值（=191.813620）。
- 实现细节：MathOpt 对 $x\le y$ 返回非正对偶，$\lambda=-\sigma$；车辆数对偶 $\mu\le 0$ 直接进割常数项 $K\mu$；
  人工变量使 SP 对任意 $y$ 可行（无需可行性割）。


In [1]:
# 环境与演示数据（28 列小池 = 25 条单客户路径 + 3 条最优路线）
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="numpy")
import sys, platform, math, time
import ortools
sys.path.insert(0, "/mnt/d/exactTest/column-generation-solvers/vrptw_solomon25/scripts")
from cg_cpsat import build_data
from ortools.math_opt.python import mathopt
print("python", platform.python_version(), "| ortools", ortools.__version__)

n, xc, yc, dem, ready, due, svc, cap, depot_due, dist, d_scaled = build_data()
K = 25
ROUTES = [(13,17,18,19,15,16,14,12), (20,24,25,23,22,21), (5,3,7,8,10,11,9,6,4,2,1)]
pool = [(i,) for i in range(1, n+1)] + ROUTES

def col_cost(p):
    c = 0.0
    prev = 0
    for j in p:
        c += dist(prev, j)
        prev = j
    return c + dist(prev, 0)

def col_mask(p):
    m = 0
    for j in p:
        m |= (1 << j)
    return m

costs = [col_cost(p) for p in pool]
masks = [col_mask(p) for p in pool]
P = len(pool)
print(f"演示池: {P} 列（25 单客户 + 3 最优路线）；最优值 191.813620（=191.81）")


# SP(y=全1) = 演示池上的完整 LP；取对偶生成第一条割
M = 10**6
yvec = [1]*P
sp = mathopt.Model()
xv = [sp.add_variable(lb=0.0, ub=float("inf"), is_integer=False, name=f"x{p}") for p in range(P)]
sv = [sp.add_variable(lb=0.0, ub=float("inf"), is_integer=False, name=f"s{i}") for i in range(1, n+1)]
covers = []
for i in range(1, n+1):
    covers.append(sp.add_linear_constraint(
        mathopt.fast_sum([xv[p] for p in range(P) if (masks[p] >> i) & 1]) + sv[i-1] >= 1.0, name=f"c{i}"))
veh = sp.add_linear_constraint(mathopt.fast_sum(xv) <= K, name="veh")
xub = [sp.add_linear_constraint(xv[p] <= float(yvec[p]), name=f"xub{p}") for p in range(P)]
sp.minimize(mathopt.fast_sum([costs[p]*xv[p] for p in range(P)]) + M*mathopt.fast_sum(sv))
res = mathopt.solve(sp, mathopt.SolverType.GLOP)
dv = res.dual_values()
pi = [max(0.0, dv[covers[i-1]]) for i in range(1, n+1)]
mu = dv[veh]
sigma = [dv[xub[p]] for p in range(P)]
alpha = sum(pi) + K*mu
lam = [-s for s in sigma]
V = res.objective_value()
print("SP(全1) 值 V =", round(V, 6), "（= 演示池 LP 松弛值）")
print("对偶: Σπ =", round(sum(pi), 4), "| Kμ =", round(K*mu, 4), "| 非零 λ 列数:", sum(1 for v in lam if v > 1e-9))
print("割:  theta + Σ λ_p y_p >= Σπ + Kμ =", round(alpha, 6))
# 强对偶：割在 y=全1 处取等
cut_at_ones = alpha - sum(lam)
print("割在 y=全1 处: theta >= alpha - Σλ*1 =", round(cut_at_ones, 6), "= V ->", abs(cut_at_ones - V) < 1e-6)
# 有效性验证：对另一个 y'（仅单客户列）解 SP，割仍应 ≤ V(y')
y2 = [1 if len(pool[p]) == 1 else 0 for p in range(P)]
sp2 = mathopt.Model()
xv2 = [sp2.add_variable(lb=0.0, ub=float("inf"), is_integer=False, name=f"x{p}") for p in range(P)]
sv2 = [sp2.add_variable(lb=0.0, ub=float("inf"), is_integer=False, name=f"s{i}") for i in range(1, n+1)]
for i in range(1, n+1):
    sp2.add_linear_constraint(
        mathopt.fast_sum([xv2[p] for p in range(P) if (masks[p] >> i) & 1]) + sv2[i-1] >= 1.0, name=f"c{i}")
sp2.add_linear_constraint(mathopt.fast_sum(xv2) <= K, name="veh")
for p in range(P):
    sp2.add_linear_constraint(xv2[p] <= float(y2[p]), name=f"xub{p}")
sp2.minimize(mathopt.fast_sum([costs[p]*xv2[p] for p in range(P)]) + M*mathopt.fast_sum(sv2))
res2 = mathopt.solve(sp2, mathopt.SolverType.GLOP)
cut_at_y2 = alpha - sum(lam[p]*y2[p] for p in range(P))
print(f"y'=仅单客户列: V(y') = {round(res2.objective_value(),4)} | 割在该 y' 处 = {round(cut_at_y2,4)} | 割有效(≤V): {cut_at_y2 <= res2.objective_value() + 1e-6}")


python 3.10.20 | ortools 9.15.6755
python 3.10.20 | ortools 9.15.6755
演示池: 28 列（25 单客户 + 3 最优路线）；最优值 191.813620（=191.81）
SP(全1) 值 V = 191.81362 （= 演示池 LP 松弛值）
对偶: Σπ = 1132.1979 | Kμ = 0.0 | 非零 λ 列数: 3
割:  theta + Σ λ_p y_p >= Σπ + Kμ = 1132.197915
割在 y=全1 处: theta >= alpha - Σλ*1 = 191.81362 = V -> True
y'=仅单客户列: V(y') = 1132.1979 | 割在该 y' 处 = 1132.1979 | 割有效(≤V): True
